In [1]:
"""
ML Orchestrator - Kaggle Notebook Version
==========================================
Complete notebook for Kaggle with GPU-accelerated GGUF inference
Multi-step planning and execution for complex ML orchestration tasks
"""

# ============================================================================
# CELL 1: Install Dependencies
# ============================================================================
print("📦 Installing dependencies...")

# Install llama-cpp-python with CUDA support (pre-built wheel)
!pip uninstall llama-cpp-python -y -q 2>/dev/null
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 -q

# Install other dependencies
!pip install huggingface-hub -q

print("✅ Installation complete!")

# ============================================================================
# CELL 2: Verify GPU
# ============================================================================
import subprocess

print("\n🎮 Checking GPU...")
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], 
                       capture_output=True, text=True)
print(f"GPU: {result.stdout.strip()}")

# ============================================================================
# CELL 3: Import and Define Core Classes
# ============================================================================
import os
import json
import pickle
import re
import warnings
from pathlib import Path
from typing import List, Dict, Any, Optional
from dataclasses import dataclass

import numpy as np
from huggingface_hub import hf_hub_download
from llama_cpp import Llama, LlamaGrammar

warnings.filterwarnings('ignore')

# Data Models
@dataclass
class ToolCall:
    name: str
    arguments: Dict[str, Any]
    reasoning: str = ""

@dataclass
class ExecutionPlan:
    steps: List[ToolCall]
    reasoning: str
    is_multi_step: bool

@dataclass
class ExecutionResult:
    tool_name: str
    inputs: Dict[str, Any]
    output: Dict[str, Any]
    success: bool
    error: Optional[str] = None


# Universal Inference Engine
class UniversalInferenceEngine:
    def __init__(self, model_path: str):
        self.model_path = Path(model_path)
        self.model_data = self._load_model(self.model_path)
        self.model_type = self._detect_type(self.model_data)

    def _load_model(self, path: Path):
        if not path.exists():
            # Demo model
            return {
                "architecture": "linear_custom",
                "params": {
                    "weights": {"age": -0.05, "bmi": -0.1, "recency_days": -0.02},
                    "intercept": 0.5,
                    "means": {"age": 30, "bmi": 25},
                    "stds": {"age": 10, "bmi": 5}
                }
            }
        with open(path, 'rb') as f:
            return pickle.load(f)

    def _detect_type(self, data: Any) -> str:
        if isinstance(data, dict) and "architecture" in data:
            return data["architecture"]
        if hasattr(data, "predict"):
            return "sklearn"
        return "unknown"

    def predict(self, inputs: Dict[str, Any]) -> Dict[str, Any]:
        try:
            if self.model_type == "linear_custom":
                return self._predict_custom_linear(inputs)
            return {"error": f"Unsupported: {self.model_type}"}
        except Exception as e:
            return {"error": str(e)}

    def _predict_custom_linear(self, inputs: Dict) -> Dict:
        params = self.model_data["params"]
        z = params.get("intercept", 0.0)
        
        for feat, val in inputs.items():
            if feat not in params["weights"]:
                continue
            val = float(val)
            if feat in params.get("means", {}):
                val = (val - params["means"][feat]) / params["stds"][feat]
            z += val * params["weights"][feat]
        
        prob = 1.0 / (1.0 + np.exp(-z))
        return {"probability": float(prob)}


# Model Registry
class ModelRegistry:
    def __init__(self, config_path: str):
        self.config_path = Path(config_path)
        self.config = json.load(open(self.config_path, 'r'))
        self.model_metadata = {}

    def load_all_models(self) -> Dict[str, callable]:
        tools = {}
        for m in self.config.get("models", []):
            try:
                engine = UniversalInferenceEngine(m.get("file_path", "dummy"))
                tools[m["id"]] = lambda eng=engine, **kwargs: eng.predict(kwargs)
                self.model_metadata[m["id"]] = {
                    "schema": m,
                    "examples": m.get("examples", []),
                    "example_outputs": m.get("example_outputs", [])
                }
            except Exception as e:
                print(f"⚠️  Error loading {m['id']}: {e}")
        return tools

    def get_all_metadata(self) -> List[Dict]:
        return list(self.model_metadata.values())
    
    def build_tools_description(self) -> str:
        tools_list = []
        for model_id, metadata in self.model_metadata.items():
            schema = metadata["schema"]
            params_desc = []
            for feat_name, feat_info in schema["feature_info"].items():
                param_type = feat_info.get("type", "number")
                description = feat_info.get("description", f"The {feat_name} value")
                params_desc.append(f"  - {feat_name} ({param_type}): {description}")
            
            tool_desc = f"""Tool: {model_id}
Description: {schema.get('description', '')}
Parameters:
{chr(10).join(params_desc)}"""
            tools_list.append(tool_desc.strip())
        
        return "\n\n".join(tools_list)


# GGUF Model Wrapper - UPDATED FOR GRAMMAR SUPPORT
class GGUFModel:
    def __init__(self, repo_id: str, filename: str, n_ctx: int = 4096, 
                 n_gpu_layers: int = -1, verbose: bool = False):
        print(f"📥 Downloading: {filename}")
        self.model_path = hf_hub_download(repo_id=repo_id, filename=filename, resume_download=True)
        print(f"🚀 Loading with GPU (ctx={n_ctx}, batch=512, Q6_K quality)...")
        
        self.llm = Llama(
            model_path=self.model_path, n_ctx=n_ctx, n_threads=4,
            n_gpu_layers=n_gpu_layers, n_batch=512, use_mlock=False, 
            use_mmap=True, verbose=False
        )
        print(f"✅ Ready!")
    
    def generate(self, prompt: str, max_tokens: int = 512, temperature: float = 0.1, 
                 grammar: Optional[LlamaGrammar] = None, stop: Optional[List[str]] = None) -> str:
        # Pass grammar to the LLM generation
        output = self.llm(
            prompt, 
            max_tokens=max_tokens, 
            temperature=temperature,
            grammar=grammar,
            echo=False, 
            stop=stop or []
        )
        return output["choices"][0]["text"].strip()


# ML Orchestrator - UPDATED: Robust Execution & Stricter Prompts
class MLOrchestrator:
    def __init__(self, model_config_path: str, gguf_repo_id: str, gguf_filename: str,
                 enable_multi_step: bool = True, verbose: bool = False):
        print("🚀 Initializing ML Orchestrator")
        self.registry = ModelRegistry(model_config_path)
        self.tools = self.registry.load_all_models()
        self.enable_multi_step = enable_multi_step
        self.verbose = verbose
        
        print(f"✅ Loaded {len(self.tools)} tools")
        self.llm = GGUFModel(repo_id=gguf_repo_id, filename=gguf_filename, verbose=verbose)
        self.tools_description = self.registry.build_tools_description()
        
        # JSON Grammar Schema
        self.json_grammar_schema = {
            "type": "object",
            "properties": {
                "is_multi_step": {"type": "boolean"},
                "reasoning": {"type": "string"},
                "steps": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "tool": {"type": "string"},
                            "arguments": {"type": "object"},
                            "reasoning": {"type": "string"}
                        },
                        "required": ["tool", "arguments", "reasoning"]
                    }
                }
            },
            "required": ["is_multi_step", "reasoning", "steps"]
        }
        self.grammar = LlamaGrammar.from_json_schema(json.dumps(self.json_grammar_schema))
        
        print("✅ Orchestrator ready!\n")
    
    def _build_planning_prompt(self, query: str) -> str:
        # FIX 1: Explicitly forbid inventing tools
        return f"""You are an AI that creates execution plans for ML tools.

Available Tools:
{self.tools_description}

Instructions:
1. Analyze the query.
2. Use ONLY the tools listed above. 
3. DO NOT invent tools for sorting, ranking, or comparison.
4. Just retrieve the data; the final summary will handle the comparison.

Query: "{query}"
JSON Plan:"""
    
    def _plan_execution(self, query: str) -> Optional[ExecutionPlan]:
        prompt = self._build_planning_prompt(query)
        
        response = self.llm.generate(
            prompt, 
            max_tokens=2048, 
            temperature=0.0,
            grammar=self.grammar
        )
        
        if self.verbose:
            print(f"📝 Raw Model Output (Guaranteed JSON):\n{response}")

        try:
            plan_dict = json.loads(response)
        except json.JSONDecodeError:
            return None
            
        steps = [ToolCall(s.get("tool", ""), s.get("arguments", {}), s.get("reasoning", ""))
                 for s in plan_dict.get("steps", [])]
        
        return ExecutionPlan(steps=steps, reasoning=plan_dict.get("reasoning", ""),
                           is_multi_step=plan_dict.get("is_multi_step", False))
    
    def _execute_step(self, step: ToolCall) -> ExecutionResult:
        if step.name not in self.tools:
            return ExecutionResult(step.name, step.arguments, {}, False, f"Tool '{step.name}' not found")
        
        try:
            clean_args = {}
            for k, v in step.arguments.items():
                try:
                    clean_args[k] = float(v)
                except:
                    clean_args[k] = v
                    
            output = self.tools[step.name](**clean_args)
            if "error" in output:
                return ExecutionResult(step.name, clean_args, output, False, output["error"])
            return ExecutionResult(step.name, clean_args, output, True)
        except Exception as e:
            return ExecutionResult(step.name, step.arguments, {}, False, str(e))
    
    def _generate_explanation(self, query: str, results: List[ExecutionResult], plan: ExecutionPlan) -> str:
        results_context = []
        for i, r in enumerate(results, 1):
            if r.success:
                prob = r.output.get("probability", "N/A")
                results_context.append(f"Step {i} ({r.tool_name}): Inputs={json.dumps(r.inputs)}, Probability={prob:.3f}")
            else:
                # Include errors in the context so the LLM knows why data is missing
                results_context.append(f"Step {i} ({r.tool_name}): FAILED - {r.error}")
        
        results_str = "\n".join(results_context)
        
        prompt = f"""Explain ML prediction results to user.

Query: "{query}"
Results:
{results_str}

Instructions:
1. Answer the query based on the successful results.
2. If a step failed, ignore it or mention it briefly.
3. Perform any necessary ranking or comparison here.

Response:"""
        
        return self.llm.generate(prompt, max_tokens=300, temperature=0.3)
    
    def predict(self, query: str) -> Dict:
        plan = self._plan_execution(query)
        if not plan or not plan.steps:
            return {"success": False, "error": "Planning failed", "query": query}
        
        if self.verbose:
            print(f"📋 Plan: {plan.reasoning} ({len(plan.steps)} steps)")
        
        results = []
        for i, step in enumerate(plan.steps, 1):
            if self.verbose:
                print(f"⚙️  Step {i}: {step.name} {step.arguments}")
            result = self._execute_step(step)
            results.append(result)
            if self.verbose:
                print(f"   {'✅' if result.success else '❌'} {result.output if result.success else result.error}")
        
        # FIX 2: Check for PARTIAL success
        # If at least one step succeeded, we proceed to explanation
        successful_steps = [r for r in results if r.success]
        
        if not successful_steps:
            return {
                "success": False, "error": "All steps failed",
                "results": [{"tool": r.tool_name, "error": r.error} for r in results]
            }
        
        # Proceed even if some steps failed (handling the hallucinated tool case)
        explanation = self._generate_explanation(query, results, plan)
        
        return {
            "success": True, 
            "query": query,
            "plan": {"is_multi_step": plan.is_multi_step, "reasoning": plan.reasoning, 
                    "steps": len(plan.steps)},
            "execution_results": [{"step": i, "tool": r.tool_name, "inputs": r.inputs, 
                                  "output": r.output, "error": r.error} for i, r in enumerate(results, 1)],
            "natural_language_response": explanation
        }

print("✅ Classes defined with Robust Logic!")

# ============================================================================
# CELL 4: Setup Configuration
# ============================================================================
config = {
    "models": [{
        "id": "donor_prediction",
        "description": "Predict blood donation likelihood based on donor characteristics",
        "file_path": "donor.pkl",
        "features": ["age", "bmi", "recency_days"],
        "feature_info": {
            "age": {"type": "number", "description": "Age of donor in years"},
            "bmi": {"type": "number", "description": "Body Mass Index"},
            "recency_days": {"type": "number", "description": "Days since last donation"}
        },
        "example_outputs": [
            "The donor has an 85% probability of donating.",
            "Likely to donate: YES (85%)"
        ]
    }]
}

Path("ml_models").mkdir(exist_ok=True)
config_path = "ml_models/config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"✅ Config saved to {config_path}")

# ============================================================================
# CELL 5: Initialize Orchestrator
# ============================================================================
orchestrator = MLOrchestrator(
    model_config_path=config_path,
    gguf_repo_id="bartowski/xLAM-7b-fc-r-GGUF",
    gguf_filename="xLAM-7b-fc-r-Q6_K.gguf",  # Higher quality with 16GB VRAM
    enable_multi_step=True,
    verbose=True
)

# ============================================================================
# CELL 6: Test Single-Step Queries
# ============================================================================
print("\n" + "="*70)
print("🧪 TEST 1: Single-Step Query")
print("="*70)

query1 = "Check donor aged 35, BMI 24, last donated 30 days ago"
result1 = orchestrator.predict(query1)

print(f"\nQuery: {query1}")
if result1['success']:
    print(f"✅ Plan: {result1['plan']['reasoning']}")
    print(f"📊 Results: {json.dumps(result1['execution_results'], indent=2)}")
    print(f"💬 Response:\n{result1['natural_language_response']}")
else:
    print(f"❌ Error: {result1.get('error')}")

# ============================================================================
# CELL 7: Test Multi-Step Queries
# ============================================================================
print("\n" + "="*70)
print("🧪 TEST 2: Multi-Step Query (Comparison)")
print("="*70)

query2 = "Compare two donors: first is 35 years old with BMI 24, second is 42 years old with BMI 28. Who is more likely to donate?"
result2 = orchestrator.predict(query2)

print(f"\nQuery: {query2}")
if result2['success']:
    print(f"✅ Plan: {result2['plan']['reasoning']}")
    print(f"   Multi-step: {result2['plan']['is_multi_step']}")
    print(f"   Steps: {result2['plan']['steps']}")
    print(f"\n📊 Execution Results:")
    for er in result2['execution_results']:
        print(f"   Step {er['step']}: {er['tool']} -> {er['output']}")
    print(f"\n💬 Response:\n{result2['natural_language_response']}")
else:
    print(f"❌ Error: {result2.get('error')}")

# ============================================================================
# CELL 8: Test Complex Multi-Step
# ============================================================================
print("\n" + "="*70)
print("🧪 TEST 3: Complex Multi-Step (Ranking)")
print("="*70)

query3 = "Analyze three donors: (1) age 28, BMI 22, recency 60 days (2) age 35, BMI 24, recency 30 days (3) age 42, BMI 28, recency 15 days. Rank them by likelihood to donate."
result3 = orchestrator.predict(query3)

print(f"\nQuery: {query3}")
if result3['success']:
    print(f"✅ Plan: {result3['plan']['reasoning']}")
    print(f"   Steps: {result3['plan']['steps']}")
    print(f"\n📊 All Results:")
    for er in result3['execution_results']:
        print(f"   {er['tool']}: {er['inputs']} -> Prob: {er['output']['probability']:.3f}")
    print(f"\n💬 Response:\n{result3['natural_language_response']}")
else:
    print(f"❌ Error: {result3.get('error')}")

# ============================================================================
# CELL 9: Custom Query
# ============================================================================
print("\n" + "="*70)
print("🎯 Try Your Own Query!")
print("="*70)

# Uncomment and modify:
# custom_query = "Your query here"
# custom_result = orchestrator.predict(custom_query)
# print(json.dumps(custom_result, indent=2))

print("\n✅ All tests complete!")

📦 Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 551.3/551.3 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.8 MB/s eta 0:00:00
✅ Installation complete!

🎮 Checking GPU...
GPU: Tesla P100-PCIE-16GB, 16384 MiB
✅ Classes defined with Robust Logic!
✅ Config saved to ml_models/config.json
🚀 Initializing ML Orchestrator
✅ Loaded 1 tools
📥 Downloading: xLAM-7b-fc-r-Q6_K.gguf


xLAM-7b-fc-r-Q6_K.gguf:   0%|          | 0.00/5.67G [00:00<?, ?B/s]

🚀 Loading with GPU (ctx=4096, batch=512, Q6_K quality)...
✅ Ready!
✅ Orchestrator ready!


🧪 TEST 1: Single-Step Query
📝 Raw Model Output (Guaranteed JSON):
{"is_multi_step": false, "reasoning": "The query is asking for the likelihood of a blood donation based on the given donor characteristics. The age, BMI, and recency_days parameters are provided.", "steps": [{"tool": "donor_prediction", "arguments": {"age": 35, "bmi": 24, "recency_days": 30}, "reasoning": "The donor_prediction tool is used to predict the likelihood of a blood donation based on the given donor characteristics."}]}
📋 Plan: The query is asking for the likelihood of a blood donation based on the given donor characteristics. The age, BMI, and recency_days parameters are provided. (1 steps)
⚙️  Step 1: donor_prediction {'age': 35, 'bmi': 24, 'recency_days': 30}
   ✅ {'probability': 0.4737740906279313}

Query: Check donor aged 35, BMI 24, last donated 30 days ago
✅ Plan: The query is asking for the likelihood of a blood d

In [2]:
# ============================================================================
# CELL 1: Install Dependencies
# ============================================================================
print("📦 Installing dependencies...")
!pip uninstall llama-cpp-python -y -q 2>/dev/null
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 -q
!pip install huggingface-hub -q
print("✅ Installation complete!")

📦 Installing dependencies...
✅ Installation complete!


Autonomous Agent Test Suite - Kaggle Version
============================================
Features:
- Dual ML Models (Likelihood + Health Risk)
- Action Tools (Schedule Appointment)
- Automated Scenario Testing & Validation


In [3]:
import random
import json
import os
from typing import List, Dict, Any, Union
from llama_cpp import Llama, LlamaGrammar
from huggingface_hub import hf_hub_download

# ============================================================================
# 1. HELPER: Clean Logger
# ============================================================================
def print_log(step, tool, thought, result):
    print("-" * 60)
    print(f"📍 STEP {step}: \033[1m{tool}\033[0m")
    print(f"💭 THOUGHT: {thought}")
    
    # Format Result for readability
    res_str = str(result)
    if len(res_str) > 200:
        print(f"🔧 RESULT:  {res_str[:200]}... [truncated]")
    else:
        print(f"🔧 RESULT:  {res_str}")
    print("-" * 60 + "\n")

# ============================================================================
# 2. THE ENVIRONMENT (Robust Tools)
# ============================================================================
class AdvancedBloodBankEnv:
    def __init__(self):
        random.seed(42)
        self.scheduled_donors = [] 
        self.inventory = {} 
        self.db = self._generate_db()
        self.valid_ids = {d["id"] for d in self.db}
        
    def _generate_db(self):
        return [
            {"id": "D_PERFECT", "name": "Alice", "blood_type": "O-", "age": 25, "weight": 70, "history": "clean"},
            {"id": "D_RISKY", "name": "Bob", "blood_type": "O-", "age": 75, "weight": 90, "history": "issues"},
            {"id": "D_WRONG_TYPE", "name": "Charlie", "blood_type": "A+", "age": 30, "weight": 70, "history": "clean"}
        ]

    def reset(self, o_negative_stock: int):
        self.inventory = {"O-": o_negative_stock, "A+": 50}
        self.scheduled_donors = []

    # --- TOOLS ---

    def check_stock(self) -> str:
        """Returns current inventory levels."""
        return json.dumps(self.inventory)

    def search_db(self, blood_type: str = "O-") -> List[Dict]:
        """Finds donors by blood type."""
        if not blood_type: blood_type = "O-"
        return [d for d in self.db if d["blood_type"] == blood_type]

    def batch_analyze_donors(self, donors: List[Dict] = None, donors_list: List[Dict] = None) -> str:
        """Analyzes a list of donors for eligibility."""
        target = donors if donors is not None else donors_list
        
        # Validation
        if target is None: return "Error: No donor list provided."
        if isinstance(target, str):
            try: target = json.loads(target)
            except: return "Error: Input must be a valid JSON list."
        if not isinstance(target, list): return "Error: Input must be a LIST."
        
        results = {}
        for d in target:
            if not isinstance(d, dict) or 'id' not in d: continue
            
            p_donate = 0.5 + (40 - abs(30 - float(d.get('age', 30)))) * 0.01
            p_health = 0.1 if d.get('history') == "issues" else 1.0
            
            if p_donate > 0.5 and p_health > 0.5:
                results[d['id']] = "ELIGIBLE"
            else:
                results[d['id']] = "REJECTED"
        return json.dumps(results)

    def schedule_appointment(self, donor_ids: Union[str, List[str]] = None, donor_id: str = None) -> str:
        """Schedules appointments for valid IDs."""
        targets = donor_ids if donor_ids is not None else donor_id
        if targets is None: return "Error: No IDs provided."
        if isinstance(targets, str): targets = [targets]
        
        logs = []
        for d_id in targets:
            if d_id not in self.valid_ids:
                logs.append(f"FAILED: {d_id} (Invalid ID)")
                continue
            if d_id in self.scheduled_donors:
                logs.append(f"SKIPPED: {d_id} (Duplicate)")
            else:
                self.scheduled_donors.append(d_id)
                logs.append(f"SUCCESS: {d_id}")
        return "; ".join(logs)

    def finish_task(self, reasoning: str = "Done") -> str:
        """Ends the session."""
        return "TASK_COMPLETE"

# ============================================================================
# 3. THE AUTONOMOUS AGENT (Config-Driven)
# ============================================================================
class AutonomousAgent:
    def __init__(self, env: AdvancedBloodBankEnv, model_repo: str, model_file: str):
        self.env = env
        
        # --- CONFIGURATION: TOOLS & EXAMPLES ---
        self.config = {
            "role": "You are a logical Blood Bank Manager. You follow instructions literally.",
            "tools": [
                {
                    "name": "check_stock",
                    "desc": "Check current inventory levels.",
                    "usage": '{"thought": "Checking stock.", "tool": "check_stock", "arguments": {}}'
                },
                {
                    "name": "search_db",
                    "desc": "Search for donors if stock < 10.",
                    "usage": '{"thought": "Stock is 5, which is less than 10. Searching.", "tool": "search_db", "arguments": {"blood_type": "O-"}}'
                },
                {
                    "name": "batch_analyze_donors",
                    "desc": "Analyze a list of donors to see who is ELIGIBLE.",
                    "usage": '{"thought": "Analyzing donors.", "tool": "batch_analyze_donors", "arguments": {"donors": [...]}}'
                },
                {
                    "name": "schedule_appointment",
                    "desc": "Schedule ONLY ELIGIBLE donors.",
                    "usage": '{"thought": "Scheduling Alice.", "tool": "schedule_appointment", "arguments": {"donor_ids": ["D_PERFECT"]}}'
                },
                {
                    "name": "finish_task",
                    "desc": "Call this when stock >= 10 OR after scheduling.",
                    "usage": '{"thought": "Stock is 50. Sufficient.", "tool": "finish_task", "arguments": {"reasoning": "Stock is sufficient."}}'
                }
            ]
        }
            
        print(f"🚀 Loading Agent Model: {model_file}...")
        path = hf_hub_download(repo_id=model_repo, filename=model_file)
        self.llm = Llama(model_path=path, n_ctx=4096, n_gpu_layers=-1, verbose=False)
        
        # Tool Mapping
        self.tools_map = {
            "check_stock": env.check_stock,
            "search_db": env.search_db,
            "batch_analyze_donors": env.batch_analyze_donors,
            "schedule_appointment": env.schedule_appointment,
            "finish_task": env.finish_task
        }

        # Grammar
        self.grammar_schema = {
            "type": "object",
            "properties": {
                "thought": {"type": "string"},
                "tool": {"type": "string"},
                "arguments": {"type": "object"}
            },
            "required": ["thought", "tool", "arguments"]
        }
        self.grammar = LlamaGrammar.from_json_schema(json.dumps(self.grammar_schema))

    def run_mission(self, objective: str, max_steps=8):
        # Crucial: Initialize history asserting IGNORANCE of the state
        history = [
            f"SYSTEM: {self.config['role']}",
            "STATUS: Session Started. Stock level is UNKNOWN.",
            "INSTRUCTION: Your FIRST action MUST be 'check_stock'."
        ]
        
        print(f"\n🤖 AGENT ACTIVE. Objective: {objective}\n")
        
        for i in range(max_steps):
            prompt = self._build_prompt(history)
            
            # Low temperature for logic
            out_str = self.llm(
                prompt, max_tokens=512, temperature=0.0, 
                repeat_penalty=1.1, grammar=self.grammar, echo=False
            )
            
            try:
                action = json.loads(out_str["choices"][0]["text"])
                tool = action["tool"]
                args = action["arguments"]
                thought = action["thought"]
            except:
                print("❌ JSON Parsing Error")
                continue

            # Execute Tool
            if tool in self.tools_map:
                try:
                    # Clean arguments
                    clean_args = {k: v for k, v in args.items() if v is not None}
                    result = self.tools_map[tool](**clean_args)
                except:
                    # Retry without args if failure
                    try: result = self.tools_map[tool]()
                    except Exception as e: result = str(e)
            else:
                result = "Error: Tool not found"

            print_log(i+1, tool, thought, result)
            
            # Append to History
            res_str = str(result)
            if len(res_str) > 500: res_str = res_str[:500] + "...(truncated)"
            
            history.append(f"ACTION: {tool}({json.dumps(args)})")
            history.append(f"RESULT: {res_str}")
            
            if tool == "finish_task":
                return
                
        print("⚠️ Max steps reached.")

    def _build_prompt(self, history):
        # 1. Dynamic Tool Documentation
        tool_docs = []
        for t in self.config["tools"]:
            doc = f"TOOL: {t['name']}\nDESC: {t['desc']}\nEXAMPLE: {t['usage']}\n"
            tool_docs.append(doc)
        
        tools_str = "\n".join(tool_docs)
        hist_txt = "\n".join(history[-8:]) 
        
        # KEY FIX: Explicit MATH comparison logic in the Rules
        return f"""You are an Autonomous Agent.

=== AVAILABLE TOOLS & EXAMPLES ===
{tools_str}

=== STRICT OPERATIONAL RULES ===
1. UNKNOWN STATE: You do not know the stock. Call 'check_stock' first.
2. INTERPRETING STOCK: 
   - Look at the result of check_stock.
   - If stock is LESS THAN 10 (e.g., 0, 5, 9): This is BAD. You MUST call 'search_db'.
   - If stock is 10 OR MORE (e.g., 10, 50, 100): This is GOOD. You MUST call 'finish_task'.
3. PROCESSING DONORS:
   - After 'search_db', pass the list to 'batch_analyze_donors'.
   - After analysis, pass ONLY 'ELIGIBLE' IDs to 'schedule_appointment'.
4. FINISHING:
   - If you scheduled donors, call 'finish_task'.
   - If stock was originally good, call 'finish_task'.

=== CURRENT HISTORY ===
{hist_txt}

Generate JSON with 'thought', 'tool', and 'arguments'.
"""

# ============================================================================
# 4. TEST RUNNER
# ============================================================================
class TestRunner:
    def __init__(self):
        self.env = AdvancedBloodBankEnv()
        self.agent = AutonomousAgent(
            self.env, 
            "bartowski/xLAM-7b-fc-r-GGUF", 
            "xLAM-7b-fc-r-Q6_K.gguf"
        )

    def run_tests(self):
        print("\n" + "="*60)
        print("🧪 STARTING VALIDATION SUITE")
        print("="*60)
        
        # TEST 1
        print("\n🔵 TEST 1: Critical Shortage (O- < 10)")
        self.env.reset(o_negative_stock=5) 
        self.agent.run_mission("Replenish O-.")
        
        scheduled = self.env.scheduled_donors
        if "D_PERFECT" in scheduled and "D_RISKY" not in scheduled:
             print("✅ PASS: Scheduled D_PERFECT, Avoided D_RISKY.")
        else:
             print(f"❌ FAIL: Inventory: {scheduled}")

        # TEST 2
        print("\n🔵 TEST 2: No Shortage (O- >= 10)")
        self.env.reset(o_negative_stock=50)
        self.agent.run_mission("Replenish O-.")
        
        if len(self.env.scheduled_donors) == 0:
            print("✅ PASS: Correctly did NOTHING.")
        else:
            print(f"❌ FAIL: Scheduled donors unnecessarily: {self.env.scheduled_donors}")

runner = TestRunner()
runner.run_tests()

🚀 Loading Agent Model: xLAM-7b-fc-r-Q6_K.gguf...

🧪 STARTING VALIDATION SUITE

🔵 TEST 1: Critical Shortage (O- < 10)

🤖 AGENT ACTIVE. Objective: Replenish O-.

------------------------------------------------------------
📍 STEP 1: check_stock
💭 THOUGHT: Checking stock.
🔧 RESULT:  {"O-": 5, "A+": 50}
------------------------------------------------------------

------------------------------------------------------------
📍 STEP 2: search_db
💭 THOUGHT: Stock is 5, which is less than 10. Searching for donors with blood type O-.
🔧 RESULT:  [{'id': 'D_PERFECT', 'name': 'Alice', 'blood_type': 'O-', 'age': 25, 'weight': 70, 'history': 'clean'}, {'id': 'D_RISKY', 'name': 'Bob', 'blood_type': 'O-', 'age': 75, 'weight': 90, 'history': 'issues... [truncated]
------------------------------------------------------------

------------------------------------------------------------
📍 STEP 3: batch_analyze_donors
💭 THOUGHT: Stock is 5, which is less than 10. Searching for donors with O- blood type.
🔧

In [4]:
# Create config
        if not os.path.exists("agent_config.json"):
             with open("agent_config.json", "w") as f:
                 json.dump({{
  "agent_name": "BloodBank_Robust_v4",
  "role": "You are a Blood Bank Manager. You follow a strict flowchart.",
  "rules": [
    "1. START: Call 'check_stock'.",
    "2. CHECK: If 'O-' is 10 or more, you MUST call 'finish_task(reasoning=\"Stock is full\")'. STOP HERE.",
    "3. IF LOW: Call 'search_db(blood_type=\"O-\")'.",
    "4. ANALYZE: Call 'batch_analyze_donors(donors=...)' with the list from step 3.",
    "5. DECISION: Call 'schedule_appointment(donor_ids=...)' for any donor marked 'ELIGIBLE'.",
    "6. END: Call 'finish_task'."
  ]
}, f)

IndentationError: unexpected indent (2351381797.py, line 2)

In [ ]:
"""
Quick Memory Cleanup for Kaggle
================================
Run this BEFORE initializing the orchestrator if you're getting OOM errors
"""

import torch
import gc
import subprocess

print("🧹 Cleaning GPU memory...")

# Clear PyTorch cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print("✅ Cleared CUDA cache")

# Force garbage collection
gc.collect()
print("✅ Ran garbage collector")

# Check current GPU usage
try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=memory.used,memory.free,memory.total', 
         '--format=csv,noheader,nounits'],
        capture_output=True, 
        text=True
    )
    
    used, free, total = map(float, result.stdout.strip().split(','))
    
    print(f"\n📊 GPU Memory Status:")
    print(f"   Used:  {used:.0f} MB ({used/total*100:.1f}%)")
    print(f"   Free:  {free:.0f} MB ({free/total*100:.1f}%)")
    print(f"   Total: {total:.0f} MB")
    
    # Estimate safe context size
    # Rule: KV cache ≈ n_ctx * 0.5 MB (for 7B model)
    safe_ctx = int(free / 0.5) - 500  # Reserve 500MB buffer
    safe_ctx = min(safe_ctx, 4096)  # Cap at 4096
    safe_ctx = max(safe_ctx, 512)   # Minimum 512
    
    print(f"\n💡 Recommendations:")
    if free > 6000:
        print(f"   ✅ You have plenty of VRAM!")
        print(f"   Recommended: n_ctx=4096, n_batch=512, Q4_K_M model")
    elif free > 4000:
        print(f"   ⚠️  Moderate VRAM available")
        print(f"   Recommended: n_ctx=2048, n_batch=256, Q4_K_M model")
    elif free > 2000:
        print(f"   ⚠️  Low VRAM - use conservative settings")
        print(f"   Recommended: n_ctx=1024, n_batch=128, Q3_K_M model")
    else:
        print(f"   ❌ Very low VRAM!")
        print(f"   Recommended: n_ctx=512, n_gpu_layers=15, Q2_K model")
        print(f"   Or consider CPU mode (n_gpu_layers=0)")
    
    print(f"\n   Estimated safe n_ctx: {safe_ctx}")
    
except Exception as e:
    print(f"⚠️  Could not check GPU status: {e}")

print("\n✅ Memory cleanup complete!")
print("Now you can initialize the orchestrator.")